In [1]:
import torch
import os
from dataclasses import dataclass
from typing import Dict

import unsloth
from datasets import load_dataset
from transformers import TextStreamer
from huggingface_hub import snapshot_download
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

PROMPT = """<image>
Extract all information from this Thai receipt. Output strictly as a JSON object matching the exact schema provided below.

Follow these strict rules:
1. Do not include markdown formatting (like ```json).
2. Do not include any conversational text.
3. If a text field is not present on the receipt, output null (without quotes).
4. If a number or financial field is not present, output 0.0.
5. If there are no individual product items listed, output an empty array [] for "items".

Schema:
{
  "processed": {
    "invoiceType": "",
    "invoiceBook": "",
    "invoiceID": "",
    "invoiceDate": "",
    "purchaseOrderID": "",
    "issuerName": "",
    "issuerAddress": "",
    "issuerTaxID": "",
    "issuerPhone": "",
    "customerName": "",
    "customerAddress": "",
    "customerTaxID": "",
    "customerPhone": "",
    "items": [
      {
        "itemNo": "",
        "itemCode": "",
        "itemName": "",
        "itemUnit": 0,
        "itemUnitCost": 0.0,
        "itemTotalCost": 0.0
      }
    ],
    "totalCost": 0.0,
    "discount": 0.0,
    "totalCostAfterDiscount": 0.0,
    "vat": 0.0,
    "grandTotal": 0.0
  }
}"""

# 0. Runtime configuration (initialization only)
MODEL_NAME = "unsloth/DeepSeek-OCR-2"

if not os.path.exists("deepseek_ocr2"):
    snapshot_download("unsloth/DeepSeek-OCR-2", local_dir = "deepseek_ocr2")
else:
    print("Model already downloaded.")

DATA_REGISTRY = {
    "thai_handwriting": {
        "name" : "iapp/thai_handwriting_dataset",
        "image_key" : "image",
        "text_key" : "text",
        "instruction" : "<image>\nOCR this image and output the Thai text."
    }
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0412 20:42:27.451000 8240 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[tensorflow|WARNING]From c:\Users\poohz\anaconda3\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.



🦥 Unsloth Zoo will now patch everything to make training faster!
Model already downloaded.
Using device: cuda


### Evaluate Deepseek-OCR Baseline Performance on Thai Hand written

In [2]:
from transformers import AutoModel

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit= True, # Enable 4-bit quantization for faster training and reduced memory usage
    auto_model= AutoModel,
    trust_remote_code=True,
    device_map="auto",
)

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Deepseekocr2 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 3080 Ti. Num GPUs = 1. Max memory: 12.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Deepseekocr2 does not support SDPA - switching to fast eager.


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


In [3]:
from datasets import load_dataset

ds = load_dataset("iapp/thai_handwriting_dataset", split="train[0:1500]")

Resolving data files:   0%|          | 0/101 [00:00<?, ?it/s]

In [4]:
image_file = 'sample_image.jpg'
output_path = 'output_results'

res = model.infer(
  tokenizer, 
  prompt = "<image>\nOCR this image and output the Thai text.", 
  image_file = image_file, 
  output_path = output_path, 
  base_size = 1024, 
  image_size = 768, 
  crop_mode = True,
  save_results = True, 
  test_compress = False
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


666666
===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]


In [5]:
ds[9]['text']

'แช่แข็งบัตรเครดิต'

### Finetune DeepSeek-OCR

In [6]:
@dataclass
class OCRFTConfig:
    model_name: str = "deepseek_ocr"
    dataset_name: str = "thai_handwriting"

    subset_rows: int = 150
    eval_ratio: float = 0.2
    seed: int = 42
    
    # LoRa
    r: int = 16
    lora_alpha: int = 16
    target_modules: list = ("q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
    lora_dropout: float = 0.05
    bias = "none"
    
    # Training
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    num_train_epochs: int = 2
    learning_rate: float = 2e-4
    logging_steps: int = 10
    max_length: int = 2048
    
    output_dir: str = "output_results"
    save_dir: str = "ocr_lora_finetuned"

In [7]:
model = FastVisionModel.get_peft_model(
    model,
    target_modules= OCRFTConfig.target_modules,
    r= OCRFTConfig.r,
    lora_alpha= OCRFTConfig.lora_alpha,
    lora_dropout= OCRFTConfig.lora_dropout,
    bias= OCRFTConfig.bias,
    random_state= OCRFTConfig.seed
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Making `model.base_model.model.model` require gradients


#### Data Prep

In [14]:
instruction = "<image>\nOCR this image and output the Thai text."

def convert_to_conversation(sample):
    """Convert a dataset sample into a conversation format expected by the model."""
    conversation = [
        {
            "role": "<|User|>",
            "content": instruction,
            "images": [sample['image']]
        },
        {
            "role": "<|Assistant|>",
            "content": sample["text"]
        },
    ]
    return {"messages": conversation}

# Load dataset
dataset = load_dataset("iapp/thai_handwriting_dataset", split="train[0:10150]")
dataset

Resolving data files:   0%|          | 0/101 [00:00<?, ?it/s]

Dataset({
    features: ['image', 'text', 'label_file'],
    num_rows: 10150
})

In [15]:
# Convert dataset to conversation format
converted_dataset = [convert_to_conversation(sample) for sample in dataset]

In [16]:
train_dataset = converted_dataset
converted_dataset[1]

{'messages': [{'role': '<|User|>',
   'content': '<image>\nOCR this image and output the Thai text.',
   'images': [<PIL.PngImagePlugin.PngImageFile image mode=RGB size=624x84>]},
  {'role': '<|Assistant|>', 'content': 'กังวล เพราะเรื่องที่ยังไม่เกิด'}]}

#### Data collator

In [17]:
import io
import math
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence

from deepseek_ocr2.modeling_deepseekocr2 import (
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)


@dataclass
class DeepSeekOCR2DataCollator:
    """Adaptive collator for receipt/invoice extraction.

    It does not force one fixed size for every image.
    Size is selected per image, then tokenized with DeepSeek helpers.
    """

    tokenizer: Any
    model: Any
    image_size: int = 768
    base_size: int = 1024
    crop_mode: bool = True
    auto_resize: bool = True
    max_dynamic_crops: int = 6
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(
        self,
        tokenizer,
        model,
        image_size: int = 768,
        base_size: int = 1024,
        crop_mode: bool = True,
        auto_resize: bool = True,
        max_dynamic_crops: int = 6,
        train_on_responses_only: bool = True,
    ):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.auto_resize = auto_resize
        self.max_dynamic_crops = max_dynamic_crops
        self.image_token_id = 128815
        self.dtype = model.dtype
        self.train_on_responses_only = train_on_responses_only

        self.image_transform = BasicImageTransform(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
            normalize=True,
        )
        self.patch_size = 16
        self.downsample_ratio = 4
        self.bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else 0
        self.pad_token_id = tokenizer.pad_token_id
        if self.pad_token_id is None:
            self.pad_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

    def _to_pil(self, image_data) -> Image.Image:
        if isinstance(image_data, Image.Image):
            return image_data.convert("RGB")
        if isinstance(image_data, dict) and "bytes" in image_data:
            return Image.open(io.BytesIO(image_data["bytes"])).convert("RGB")
        if isinstance(image_data, str):
            return Image.open(image_data).convert("RGB")
        raise ValueError(f"Unsupported image format: {type(image_data)}")

    def _choose_sizes(self, image: Image.Image) -> Tuple[int, int, bool]:
        if not self.auto_resize:
            return self.base_size, self.image_size, self.crop_mode

        long_side = max(image.size)

        # DeepSeek encoder supports query sizes mapped from base_size 768 or 1024 only.
        if long_side <= 900:
            return 768, 768, False
        return 1024, 768, True

    def _image_tensors_and_tokens(self, image: Image.Image):
        local_base, local_image_size, use_crops = self._choose_sizes(image)

        images_crop_list = []
        width_crop_num, height_crop_num = 1, 1

        if use_crops and max(image.size) > local_image_size:
            images_crop_raw, crop_ratio = dynamic_preprocess(
                image,
                min_num=2,
                max_num=self.max_dynamic_crops,
                image_size=local_image_size,
                use_thumbnail=False,
            )
            width_crop_num, height_crop_num = crop_ratio

            for crop_img in images_crop_raw:
                images_crop_list.append(self.image_transform(crop_img).to(self.dtype))

        global_view = ImageOps.pad(
            image,
            (local_base, local_base),
            color=tuple(int(x * 255) for x in self.image_transform.mean),
        )
        images_ori = self.image_transform(global_view).to(self.dtype).unsqueeze(0)

        if images_crop_list:
            images_crop = torch.stack(images_crop_list, dim=0)
        else:
            images_crop = torch.zeros((1, 3, local_base, local_base), dtype=self.dtype)

        images_spatial_crop = torch.tensor([[width_crop_num, height_crop_num]], dtype=torch.long)

        # IMPORTANT: Must match DeepSeek-OCR infer() token layout exactly.
        num_queries = math.ceil((local_image_size // self.patch_size) / self.downsample_ratio)
        num_queries_base = math.ceil((local_base // self.patch_size) / self.downsample_ratio)

        # Global: Q_base^2 + 1 separator
        tokenized_image = ([self.image_token_id] * num_queries_base) * num_queries_base
        tokenized_image += [self.image_token_id]

        # Local crops: (Q_local * w) * (Q_local * h), no row separators
        if width_crop_num > 1 or height_crop_num > 1:
            tokenized_image += ([self.image_token_id] * (num_queries * width_crop_num)) * (
                num_queries * height_crop_num
            )

        return images_ori, images_crop, images_spatial_crop, tokenized_image

    def _tokenize_sample(self, messages: List[Dict], tokenized_image: List[int]):
        if len(messages) < 2:
            raise ValueError("Each sample must contain user and assistant messages.")

        user_text = str(messages[0].get("content", ""))
        if "<image>" not in user_text:
            user_text = f"<image>\n{user_text}"

        assistant_text = str(messages[1].get("content", "")).strip()
        eos_text = self.tokenizer.eos_token if self.tokenizer.eos_token is not None else ""

        text_splits = user_text.split("<image>")

        tokenized_str = [self.bos_id]
        images_seq_mask = [False]

        for i, text_sep in enumerate(text_splits):
            t = text_encode(self.tokenizer, text_sep, bos=False, eos=False)
            tokenized_str.extend(t)
            images_seq_mask.extend([False] * len(t))

            if i < len(text_splits) - 1:
                tokenized_str.extend(tokenized_image)
                images_seq_mask.extend([True] * len(tokenized_image))

        prompt_token_count = len(tokenized_str)

        assistant_payload = f"{assistant_text} {eos_text}".strip()
        assistant_tokens = text_encode(self.tokenizer, assistant_payload, bos=False, eos=False)
        tokenized_str.extend(assistant_tokens)
        images_seq_mask.extend([False] * len(assistant_tokens))

        return (
            torch.tensor(tokenized_str, dtype=torch.long),
            torch.tensor(images_seq_mask, dtype=torch.bool),
            prompt_token_count,
        )

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_data = []

        for feature in features:
            messages = feature["messages"]
            image_list = messages[0].get("images", [])
            if not image_list:
                raise ValueError("No image found in sample user message.")

            pil_image = self._to_pil(image_list[0])
            images_ori, images_crop, images_spatial_crop, tokenized_image = self._image_tensors_and_tokens(pil_image)
            input_ids, images_seq_mask, prompt_token_count = self._tokenize_sample(messages, tokenized_image)

            batch_data.append(
                {
                    "input_ids": input_ids,
                    "images_seq_mask": images_seq_mask,
                    "images_ori": images_ori,
                    "images_crop": images_crop,
                    "images_spatial_crop": images_spatial_crop,
                    "prompt_token_count": prompt_token_count,
                }
            )

        input_ids_list = [item["input_ids"] for item in batch_data]
        images_seq_mask_list = [item["images_seq_mask"] for item in batch_data]
        prompt_token_counts = [item["prompt_token_count"] for item in batch_data]

        input_ids = pad_sequence(
            input_ids_list,
            batch_first=True,
            padding_value=self.pad_token_id,
        )
        images_seq_mask = pad_sequence(
            images_seq_mask_list,
            batch_first=True,
            padding_value=False,
        )

        labels = input_ids.clone()
        labels[labels == self.pad_token_id] = -100
        labels[images_seq_mask] = -100

        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100

        attention_mask = (input_ids != self.pad_token_id).long()

        images_batch = [(item["images_crop"], item["images_ori"]) for item in batch_data]
        images_spatial_crop = torch.cat([item["images_spatial_crop"] for item in batch_data], dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }


print("DeepSeekOCRDataCollator is ready (adaptive receipt/invoice mode).")

DeepSeekOCRDataCollator is ready (adaptive receipt/invoice mode).


In [18]:
# from transformers import Trainer, TrainingArguments
# from unsloth import is_bf16_supported

# FastVisionModel.for_training(model) # Enable for training
# data_collator = DeepSeekOCR2DataCollator(
#     tokenizer=tokenizer,
#     model=model,
#     image_size=768,
#     base_size=1024,
#     crop_mode=True,
#     train_on_responses_only=True
# )
# trainer = Trainer(
#     model=model,
#     train_dataset=converted_dataset,
#     data_collator=data_collator,
#     args=TrainingArguments(
#         per_device_train_batch_size=2,
#         gradient_accumulation_steps=4,
#         warmup_steps=5,
#         max_steps=60,
#         # num_train_epcohs=1, Set this instaed of max_steps for full training runs
#         learning_rate=2e-4,
#         logging_steps=1,
#         optim="adamw_8bit",
#         seed=42,
#         no_cuda=False,
#         fp16= not is_bf16_supported(), # Use fp16 if bf16 is not supported
#         bf16= is_bf16_supported(), # Use bf16 if supported
#         output_dir="output_results",
#         report_to="none",
#         dataloader_num_workers=2,
#         # You MUST put the below items for vision finetuning:
#         remove_unused_columns=False,
#     )
# )


#### Training the Model: Phase 1

Dataset: `iapp/thai_handwriting_dataset` (`train[0:10150]`, 10,150 samples)

```python
# ---- Training knobs (receipt/invoice tuned) ----
USE_MAX_STEPS = False          # False = epoch-based, True = step-based
NUM_TRAIN_EPOCHS = 3
MAX_TRAIN_STEPS = 60

# Batch & optimization
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4

# Logging & output
OUTPUT_DIR = "outputs/thai_receipt_lora"
DATALOADER_NUM_WORKERS = 0     # Better stability on Windows notebooks
LOG_EVERY_STEPS = 1            # Force visible logs every N optimizer steps

# Adaptive image preprocessing (not fixed to one size)
TRAIN_IMAGE_SIZE = 768
TRAIN_BASE_SIZE = 1024
TRAIN_CROP_MODE = True
AUTO_RESIZE_BY_IMAGE = True
MAX_DYNAMIC_CROPS = 6          # Keep aligned with dynamic_preprocess defaults
```

Note: If `USE_MAX_STEPS` is `False`, training uses `NUM_TRAIN_EPOCHS`. If `True`, it uses `MAX_TRAIN_STEPS`.

In [19]:
import math
import os
import torch
from transformers import Trainer, TrainingArguments, TrainerCallback
from unsloth import is_bf16_supported

# ---- Debug-friendly CUDA error reporting ----
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# ---- Training knobs (receipt/invoice tuned) ----
USE_MAX_STEPS = False          # False = epoch-based, True = step-based
NUM_TRAIN_EPOCHS = 3
MAX_TRAIN_STEPS = 60
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
OUTPUT_DIR = "outputs/thai_receipt_lora"
DATALOADER_NUM_WORKERS = 0     # Better stability on Windows notebooks
LOG_EVERY_STEPS = 1            # Force visible logs every N optimizer steps

# Adaptive image preprocessing (NOT fixed to one size)
TRAIN_IMAGE_SIZE = 768
TRAIN_BASE_SIZE = 1024
TRAIN_CROP_MODE = True
AUTO_RESIZE_BY_IMAGE = True
MAX_DYNAMIC_CROPS = 6          # Keep aligned with model's dynamic_preprocess defaults

# Add LoRA adapters once (skip if already wrapped)
if not hasattr(model, "peft_config"):
    model = FastVisionModel.get_peft_model(
        model,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )

FastVisionModel.for_training(model)

data_collator = DeepSeekOCR2DataCollator(
    tokenizer=tokenizer,
    model=model,
    image_size=TRAIN_IMAGE_SIZE,
    base_size=TRAIN_BASE_SIZE,
    auto_resize=AUTO_RESIZE_BY_IMAGE,
    max_dynamic_crops=MAX_DYNAMIC_CROPS,
    crop_mode=TRAIN_CROP_MODE,
    train_on_responses_only=True,
)

def _move_batch_to_device(batch, model_ref):
    try:
        device = next(model_ref.parameters()).device
    except StopIteration:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    return {
        "input_ids": batch["input_ids"].to(device),
        "attention_mask": batch["attention_mask"].to(device),
        "labels": batch["labels"].to(device),
        "images": [(crops.to(device), ori.to(device)) for crops, ori in batch["images"]],
        "images_seq_mask": batch["images_seq_mask"].to(device),
        "images_spatial_crop": batch["images_spatial_crop"].to(device),
    }


# ---- Preflight check: catch alignment issues before Trainer loop ----
print("[sanity] running one-sample forward check...", flush=True)
try:
    model.eval()
    sample_batch = data_collator([train_dataset[0]])
    sample_batch = _move_batch_to_device(sample_batch, model)

    with torch.no_grad():
        sample_out = model(**sample_batch)

    sample_loss = float(sample_out.loss.detach().cpu()) if sample_out.loss is not None else float("nan")
    print(f"[sanity] forward pass OK. sample_loss={sample_loss:.6f}", flush=True)
except Exception as e:
    print(f"[sanity] forward check FAILED: {type(e).__name__}: {e}", flush=True)
    raise
finally:
    model.train()


effective_batch_size = PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
approx_steps_per_epoch = max(1, math.ceil(len(train_dataset) / effective_batch_size))
planned_steps = MAX_TRAIN_STEPS if USE_MAX_STEPS else approx_steps_per_epoch * NUM_TRAIN_EPOCHS

training_args = dict(
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=5,
    learning_rate=LEARNING_RATE,
    logging_steps=LOG_EVERY_STEPS,
    logging_strategy="steps",
    logging_first_step=True,
    log_level="info",
    disable_tqdm=True,  # Use explicit print logs below so progress is always visible
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="linear",
    seed=3407,
    fp16=not is_bf16_supported(),
    bf16=is_bf16_supported(),
    output_dir=OUTPUT_DIR,
    report_to="none",
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    remove_unused_columns=False,
)

if USE_MAX_STEPS:
    training_args["max_steps"] = MAX_TRAIN_STEPS
else:
    training_args["num_train_epochs"] = NUM_TRAIN_EPOCHS

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=train_dataset,
    args=TrainingArguments(**training_args),
)


class NotebookProgressCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        print("[train] started", flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        loss = logs.get("loss")
        lr = logs.get("learning_rate")
        if loss is not None:
            if lr is not None:
                print(f"[train] step={state.global_step} loss={loss:.6f} lr={lr:.2e}", flush=True)
            else:
                print(f"[train] step={state.global_step} loss={loss:.6f}", flush=True)

    def on_train_end(self, args, state, control, **kwargs):
        print("[train] finished", flush=True)


trainer.add_callback(NotebookProgressCallback())

print("Trainer is ready.", flush=True)
print(f"Samples: {len(train_dataset)}", flush=True)
print(f"Effective batch size: {effective_batch_size}", flush=True)
print(f"Approx steps/epoch: {approx_steps_per_epoch}", flush=True)
print(f"Planned optimizer steps: {planned_steps}", flush=True)
if USE_MAX_STEPS:
    print(f"Training mode: steps ({MAX_TRAIN_STEPS} max steps)", flush=True)
else:
    print(f"Training mode: epochs ({NUM_TRAIN_EPOCHS} epochs)", flush=True)
print(
    f"Adaptive image mode: auto_resize={AUTO_RESIZE_BY_IMAGE}, "
    f"base_size={TRAIN_BASE_SIZE}, image_size={TRAIN_IMAGE_SIZE}, crop_mode={TRAIN_CROP_MODE}",
    flush=True,
)

# Start training
trainer_stats = trainer.train()

# Save adapter and tokenizer for later import testing
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved model artifacts to:", OUTPUT_DIR, flush=True)
print("Final metrics:", trainer_stats.metrics, flush=True)

{'loss': 1.0444, 'grad_norm': 2.84312105178833, 'learning_rate': 0.00014908660796425288, 'epoch': 0.7657142857142857}
[train] step=1943 loss=1.044400 lr=1.49e-04
{'loss': 0.8796, 'grad_norm': 2.151566982269287, 'learning_rate': 0.00014906032330135366, 'epoch': 0.7661083743842364}
[train] step=1944 loss=0.879600 lr=1.49e-04
{'loss': 1.379, 'grad_norm': 4.752017974853516, 'learning_rate': 0.00014903403863845445, 'epoch': 0.7665024630541872}
[train] step=1945 loss=1.379000 lr=1.49e-04
{'loss': 1.1329, 'grad_norm': 2.892103672027588, 'learning_rate': 0.00014900775397555527, 'epoch': 0.766896551724138}
[train] step=1946 loss=1.132900 lr=1.49e-04
{'loss': 1.5465, 'grad_norm': 3.1822447776794434, 'learning_rate': 0.00014898146931265605, 'epoch': 0.7672906403940887}
[train] step=1947 loss=1.546500 lr=1.49e-04
{'loss': 1.9404, 'grad_norm': 4.997806549072266, 'learning_rate': 0.0001489551846497569, 'epoch': 0.7676847290640394}
[train] step=1948 loss=1.940400 lr=1.49e-04
{'loss': 0.7881, 'grad_no

Saving model checkpoint to outputs/thai_receipt_lora\checkpoint-2000
loading configuration file config.json from cache at C:\Users\poohz\.cache\huggingface\hub\models--unsloth--DeepSeek-OCR-2\snapshots\c3641f7390767277f34af8e5e6e87f283fde8501\config.json
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
Model config DeepseekOCR2Config {
  "architectures": [
    "DeepseekOCR2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "modeling_deepseekocr2.DeepseekOCR2Config",
    "AutoModel": "modeling_deepseekocr2.DeepseekOCR2ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 0,
  "candidate_resolutions": [
    [
      1024,
      1024
    ]
  ],
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "first_k_dense_replace": 1,
  "global_view_pos": "head",
  "head_dim": 0,
  "hidden_act": "silu",
  "hidden_size": 1280,
 

{'loss': 1.1001, 'grad_norm': 3.4228198528289795, 'learning_rate': 0.00014756209751609937, 'epoch': 0.7885714285714286}
[train] step=2001 loss=1.100100 lr=1.48e-04
{'loss': 1.6, 'grad_norm': 2.5171682834625244, 'learning_rate': 0.00014753581285320018, 'epoch': 0.7889655172413793}
[train] step=2002 loss=1.600000 lr=1.48e-04
{'loss': 1.1731, 'grad_norm': 3.0914783477783203, 'learning_rate': 0.00014750952819030097, 'epoch': 0.78935960591133}
[train] step=2003 loss=1.173100 lr=1.48e-04
{'loss': 0.2853, 'grad_norm': 1.7408618927001953, 'learning_rate': 0.00014748324352740176, 'epoch': 0.7897536945812808}
[train] step=2004 loss=0.285300 lr=1.47e-04
{'loss': 0.7917, 'grad_norm': 1.875624179840088, 'learning_rate': 0.00014745695886450257, 'epoch': 0.7901477832512315}
[train] step=2005 loss=0.791700 lr=1.47e-04
{'loss': 1.8857, 'grad_norm': 3.0528299808502197, 'learning_rate': 0.00014743067420160336, 'epoch': 0.7905418719211823}
[train] step=2006 loss=1.885700 lr=1.47e-04
{'loss': 0.7197, 'grad

Saving model checkpoint to outputs/thai_receipt_lora\checkpoint-2500
loading configuration file config.json from cache at C:\Users\poohz\.cache\huggingface\hub\models--unsloth--DeepSeek-OCR-2\snapshots\c3641f7390767277f34af8e5e6e87f283fde8501\config.json
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
Model config DeepseekOCR2Config {
  "architectures": [
    "DeepseekOCR2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "modeling_deepseekocr2.DeepseekOCR2Config",
    "AutoModel": "modeling_deepseekocr2.DeepseekOCR2ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 0,
  "candidate_resolutions": [
    [
      1024,
      1024
    ]
  ],
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "first_k_dense_replace": 1,
  "global_view_pos": "head",
  "head_dim": 0,
  "hidden_act": "silu",
  "hidden_size": 1280,
 

{'loss': 0.9977, 'grad_norm': 2.081024408340454, 'learning_rate': 0.0001344197660665002, 'epoch': 0.985615763546798}
[train] step=2501 loss=0.997700 lr=1.34e-04
{'loss': 1.718, 'grad_norm': 3.5364151000976562, 'learning_rate': 0.000134393481403601, 'epoch': 0.9860098522167488}
[train] step=2502 loss=1.718000 lr=1.34e-04
{'loss': 1.7276, 'grad_norm': 3.0939090251922607, 'learning_rate': 0.0001343671967407018, 'epoch': 0.9864039408866995}
[train] step=2503 loss=1.727600 lr=1.34e-04
{'loss': 0.6474, 'grad_norm': 2.1185967922210693, 'learning_rate': 0.00013434091207780263, 'epoch': 0.9867980295566502}
[train] step=2504 loss=0.647400 lr=1.34e-04
{'loss': 1.0861, 'grad_norm': 2.1497323513031006, 'learning_rate': 0.0001343146274149034, 'epoch': 0.987192118226601}
[train] step=2505 loss=1.086100 lr=1.34e-04
{'loss': 1.6664, 'grad_norm': 3.0754122734069824, 'learning_rate': 0.00013428834275200423, 'epoch': 0.9875862068965517}
[train] step=2506 loss=1.666400 lr=1.34e-04
{'loss': 1.5087, 'grad_no

...

{'loss': 0.2612, 'grad_norm': 1.9836875200271606, 'learning_rate': 2.9964515705086084e-06, 'epoch': 2.955665024630542}
[train] step=7501 loss=0.261200 lr=3.00e-06
{'loss': 0.193, 'grad_norm': 1.3492625951766968, 'learning_rate': 2.97016690760941e-06, 'epoch': 2.9560591133004928}
[train] step=7502 loss=0.193000 lr=2.97e-06
{'loss': 0.3132, 'grad_norm': 1.8012133836746216, 'learning_rate': 2.9438822447102116e-06, 'epoch': 2.9564532019704433}
[train] step=7503 loss=0.313200 lr=2.94e-06
{'loss': 0.0519, 'grad_norm': 0.9972534775733948, 'learning_rate': 2.917597581811013e-06, 'epoch': 2.956847290640394}
[train] step=7504 loss=0.051900 lr=2.92e-06
{'loss': 0.6453, 'grad_norm': 2.8139121532440186, 'learning_rate': 2.891312918911815e-06, 'epoch': 2.9572413793103447}
[train] step=7505 loss=0.645300 lr=2.89e-06
{'loss': 0.0236, 'grad_norm': 0.5943114757537842, 'learning_rate': 2.8650282560126167e-06, 'epoch': 2.9576354679802956}
[train] step=7506 loss=0.023600 lr=2.87e-06
{'loss': 0.0769, 'grad_

Saving model checkpoint to outputs/thai_receipt_lora\checkpoint-7614
loading configuration file config.json from cache at C:\Users\poohz\.cache\huggingface\hub\models--unsloth--DeepSeek-OCR-2\snapshots\c3641f7390767277f34af8e5e6e87f283fde8501\config.json
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
Model config DeepseekOCR2Config {
  "architectures": [
    "DeepseekOCR2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "modeling_deepseekocr2.DeepseekOCR2Config",
    "AutoModel": "modeling_deepseekocr2.DeepseekOCR2ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 0,
  "candidate_resolutions": [
    [
      1024,
      1024
    ]
  ],
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "first_k_dense_replace": 1,
  "global_view_pos": "head",
  "head_dim": 0,
  "hidden_act": "silu",
  "hidden_size": 1280,
 

{'train_runtime': 139441.7396, 'train_samples_per_second': 0.218, 'train_steps_per_second': 0.055, 'train_loss': 0.7026318184552736, 'epoch': 3.0}
[train] finished


Saving model checkpoint to outputs/thai_receipt_lora
loading configuration file config.json from cache at C:\Users\poohz\.cache\huggingface\hub\models--unsloth--DeepSeek-OCR-2\snapshots\c3641f7390767277f34af8e5e6e87f283fde8501\config.json
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
Model config DeepseekOCR2Config {
  "architectures": [
    "DeepseekOCR2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "modeling_deepseekocr2.DeepseekOCR2Config",
    "AutoModel": "modeling_deepseekocr2.DeepseekOCR2ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 0,
  "candidate_resolutions": [
    [
      1024,
      1024
    ]
  ],
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "first_k_dense_replace": 1,
  "global_view_pos": "head",
  "head_dim": 0,
  "hidden_act": "silu",
  "hidden_size": 1280,
  "initializer_ra

Saved model artifacts to: outputs/thai_receipt_lora
Final metrics: {'train_runtime': 139441.7396, 'train_samples_per_second': 0.218, 'train_steps_per_second': 0.055, 'train_loss': 0.7026318184552736, 'epoch': 3.0}


#### Next test model Phase 1 with raise of dataset

In [65]:
import os
from pathlib import Path
import torch
from transformers import AutoModel
from unsloth import FastVisionModel

LORA_OUTPUT_DIR = Path("outputs/thai_receipt_lora")

# Recommended decoding settings
TEMPERATURE = 0.7
MAX_TOKENS = 8192
NGRAM_SIZE = 30
WINDOW_SIZE = 90  # Informational only in this local infer path (not a runtime generate arg).

def _resolve_adapter_dir(base_dir: Path) -> Path:
    if (base_dir / "adapter_config.json").exists():
        return base_dir

    checkpoints = sorted(
        [
            p
            for p in base_dir.glob("checkpoint-*")
            if (p / "adapter_config.json").exists()
        ],
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
    )

    if checkpoints:
        return checkpoints[-1]

    raise FileNotFoundError(f"No LoRA adapter found under: {base_dir}")

def _has_meta_tensors(model_obj) -> bool:
    for _, param in model_obj.named_parameters():
        if param.device.type == "meta":
            return True
    for _, buf in model_obj.named_buffers():
        if buf.device.type == "meta":
            return True
    return False

def _load_lora_for_inference(adapter_dir: Path):
    common_kwargs = dict(
        model_name=str(adapter_dir),
        auto_model=AutoModel,
        trust_remote_code=True,
        unsloth_force_compile=False,
        use_gradient_checkpointing="unsloth",
        low_cpu_mem_usage=False,
    )

    # Try 4-bit first to reduce VRAM pressure and avoid meta/offload issues.
    attempts = [
        {"load_in_4bit": True, "device_map": "auto", "label": "4bit-auto"},
        {"load_in_4bit": False, "device_map": "auto", "label": "16bit-auto"},
    ]

    last_error = None
    for cfg in attempts:
        try:
            print(f"Loading with profile: {cfg['label']}")
            model_obj, tokenizer_obj = FastVisionModel.from_pretrained(
                **common_kwargs,
                load_in_4bit=cfg["load_in_4bit"],
                device_map=cfg["device_map"],
            )

            if _has_meta_tensors(model_obj):
                raise RuntimeError("Model still contains meta tensors after load.")

            FastVisionModel.for_inference(model_obj)
            return model_obj, tokenizer_obj
        except Exception as e:
            last_error = e
            print(f"Load failed for {cfg['label']}: {type(e).__name__}: {e}")
            if "model_obj" in locals():
                del model_obj
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    raise RuntimeError(
        f"Unable to load LoRA model without meta tensors from {adapter_dir}. Last error: {last_error}"
    )

def _apply_generation_settings(model_obj, temperature: float, max_tokens: int, ngram_size: int):
    base_generate = model_obj.generate

    def _generate_with_cfg(*args, **kwargs):
        # Force preferred decoding behavior even if infer() passes its own defaults.
        kwargs["temperature"] = temperature
        kwargs["max_new_tokens"] = max_tokens
        kwargs["no_repeat_ngram_size"] = ngram_size
        return base_generate(*args, **kwargs)

    model_obj.generate = _generate_with_cfg

adapter_dir = _resolve_adapter_dir(LORA_OUTPUT_DIR)
print(f"Loading adapter from: {adapter_dir}")

model, tokenizer = _load_lora_for_inference(adapter_dir)
_apply_generation_settings(model, TEMPERATURE, MAX_TOKENS, NGRAM_SIZE)


prompt = "<image>\nFree OCR."
image_file = "test_sample_image.jpg"
output_path = "output_results"

if not Path(image_file).exists():
    raise FileNotFoundError(f"Image not found: {image_file}")

os.makedirs(output_path, exist_ok=True)

res = model.infer(
    tokenizer,
    prompt=prompt,
    image_file=image_file,
    output_path=output_path,
    image_size=TRAIN_IMAGE_SIZE if "TRAIN_IMAGE_SIZE" in globals() else 768,
    base_size=TRAIN_BASE_SIZE if "TRAIN_BASE_SIZE" in globals() else 1024,
    crop_mode=TRAIN_CROP_MODE if "TRAIN_CROP_MODE" in globals() else True,
    save_results=True,
    test_compress=False,
)

Loading adapter from: outputs\thai_receipt_lora
Loading with profile: 4bit-auto


loading configuration file config.json from cache at C:\Users\poohz\.cache\huggingface\hub\models--unsloth--DeepSeek-OCR-2\snapshots\c3641f7390767277f34af8e5e6e87f283fde8501\config.json
loading configuration file config.json from cache at C:\Users\poohz\.cache\huggingface\hub\models--unsloth--DeepSeek-OCR-2\snapshots\c3641f7390767277f34af8e5e6e87f283fde8501\config.json
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
Model config DeepseekOCR2Config {
  "architectures": [
    "DeepseekOCR2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "modeling_deepseekocr2.DeepseekOCR2Config",
    "AutoModel": "modeling_deepseekocr2.DeepseekOCR2ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 0,
  "candidate_resolutions": [
    [
      1024,
      1024
    ]
  ],
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "firs

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Deepseekocr2 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 3080 Ti. Num GPUs = 1. Max memory: 12.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Deepseekocr2 does not support SDPA - switching to fast eager.


loading configuration file config.json from cache at C:\Users\poohz\.cache\huggingface\hub\models--unsloth--DeepSeek-OCR-2\snapshots\c3641f7390767277f34af8e5e6e87f283fde8501\config.json
loading configuration file config.json from cache at C:\Users\poohz\.cache\huggingface\hub\models--unsloth--DeepSeek-OCR-2\snapshots\c3641f7390767277f34af8e5e6e87f283fde8501\config.json
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
Model config DeepseekOCR2Config {
  "architectures": [
    "DeepseekOCR2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "modeling_deepseekocr2.DeepseekOCR2Config",
    "AutoModel": "modeling_deepseekocr2.DeepseekOCR2ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 0,
  "candidate_resolutions": [
    [
      1024,
      1024
    ]
  ],
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "firs

ที่ที่เชื่อไม่อยู่นั้นก็เลยขออำนาจเองเพราะเห็นไม่ถูกมึดกลัวเราด้วยคนครอน 
===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]
